<!-- NOTEBOOK_METADATA source: "Jupyter Notebook" title: "Trace Kiro CLI Sessions with Langfuse" sidebarTitle: "Kiro CLI" logo: "/images/integrations/kiro_icon.svg" description: "Add LLM observability to Kiro CLI coding sessions with Langfuse. Trace agent prompts, tool calls, and completions automatically via Kiro CLI hooks." category: "Integrations" -->

# Trace Kiro CLI with Langfuse

This guide shows you how to integrate [Langfuse](https://langfuse.com) tracing with [Kiro CLI](https://kiro.dev/cli/) so every coding session — agent prompts, tool calls, and completion status — is captured automatically using [Kiro CLI hooks](https://kiro.dev/docs/cli/hooks/).

> **What is Kiro CLI?** [Kiro CLI](https://kiro.dev/cli/) is the terminal-based AI coding agent from [Kiro](https://kiro.dev). It runs an AI agent in your shell that can read and write files, run commands, and call MCP tools to help you build software.

> **What is Langfuse?** [Langfuse](https://github.com/langfuse/langfuse) is an open-source LLM engineering platform that helps you trace, debug, and evaluate your LLM applications.

> **Note**: This integration is community-maintained at [`dhanpraja231/kiro-cli-langfuse`](https://github.com/dhanpraja231/kiro-cli-langfuse). It uses Kiro CLI's built-in hook system, so no changes to Kiro itself are required.

## How it works

Kiro CLI exposes five hook events that fire over the lifecycle of an agent session. The integration ships a small Node.js handler that subscribes to all five and forwards them to Langfuse via the JS SDK.

| Kiro CLI hook | What it traces in Langfuse |
| --- | --- |
| `agentSpawn` | Agent initialization (event) |
| `userPromptSubmit` | User prompts and queries (generation) |
| `preToolUse` | Tool invocation before execution (span start) |
| `postToolUse` | Tool result and duration (span end) |
| `stop` | Agent completion with status score (event + score) |

Each conversation becomes one trace, sessions are grouped by working directory (`cwd`), and tools/models are surfaced as tags.

<!-- STEPS_START -->
## Prerequisites

Before you begin, ensure you have:

1. [Kiro CLI](https://kiro.dev/cli/) installed and working in your terminal.
2. [Node.js](https://nodejs.org/) v18 or newer.
3. A Langfuse account ([sign up for free](https://cloud.langfuse.com)) or a [self-hosted](https://langfuse.com/self-hosting) Langfuse instance.
4. Langfuse API keys from your project settings.

## Step 1: Add the integration to your project

Clone or download the [`kiro-cli-langfuse`](https://github.com/dhanpraja231/kiro-cli-langfuse) repository, then copy the agent definition and hook handler into the project where you use Kiro CLI:

```bash
git clone https://github.com/dhanpraja231/kiro-cli-langfuse.git

# from the root of the project you want to instrument
cp -r kiro-cli-langfuse/.kiro/agents/  .kiro/agents/
cp -r kiro-cli-langfuse/hooks/         hooks/
```

This adds two things to your project:

- `.kiro/agents/langfuse-observer.json` — a Kiro CLI agent definition that wires every hook event to the Node.js handler.
- `hooks/` — the hook handler, the Langfuse JS SDK wrapper, and per-event handlers.

## Step 2: Install dependencies

Install the handler's dependencies (the Langfuse JS SDK and a couple of small helpers):

```bash
cd hooks && npm install
```

## Step 3: Configure Langfuse credentials

Copy the example env file and fill in your Langfuse keys:

```bash
cp .env.example .env
```

```bash
# .env
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...

# 🇪🇺 EU region
LANGFUSE_BASE_URL=https://cloud.langfuse.com
# 🇺🇸 US region
# LANGFUSE_BASE_URL=https://us.cloud.langfuse.com
```

> **Note**: For self-hosted Langfuse, set `LANGFUSE_BASE_URL` to your own URL (e.g., `http://localhost:3000`).

## Step 4: Activate the Langfuse-observer agent

The hooks are defined inside `.kiro/agents/langfuse-observer.json`. The agent inherits `"tools": ["*"]`, so it keeps full access to file read/write, shell, AWS, and MCP tools while emitting traces in the background.

Switch to it inside Kiro CLI:

```bash
/agent swap langfuse-observer
```

Alternatively, merge the `hooks` field into your existing agent config so any agent you use is traced:

```json
{
  "name": "my-agent",
  "hooks": {
    "agentSpawn":       [{ "command": "node hooks/hook-handler.js" }],
    "userPromptSubmit": [{ "command": "node hooks/hook-handler.js" }],
    "preToolUse":       [{ "command": "node hooks/hook-handler.js" }],
    "postToolUse":      [{ "command": "node hooks/hook-handler.js" }],
    "stop":             [{ "command": "node hooks/hook-handler.js" }]
  }
}
```

## Step 5 (optional): Filter which tools are traced

By default every tool invocation is captured. To trace only specific tools, add a `matcher` to the `preToolUse`/`postToolUse` hooks in `.kiro/agents/langfuse-observer.json`:

```json
{
  "hooks": {
    "preToolUse": [
      {
        "matcher": "fs_write",
        "command": "node hooks/hook-handler.js"
      }
    ]
  }
}
```

Common matchers:

- `fs_write` / `fs_read` — file write/read tools
- `execute_bash` — shell commands
- `use_aws` — AWS CLI tool
- `@git` — all tools from the git MCP server (use `@git/status` for a single MCP tool)
- `*` — all tools (default)

## Step 6: View traces in Langfuse

Start a Kiro CLI session as usual and ask the agent to do something. Once you're back in [Langfuse](https://cloud.langfuse.com), you'll see one trace per conversation, with the user prompt as a generation, every tool call as a span, and a completion score on the final `stop` event. Sessions are grouped by working directory so all traces from the same project show up together.

Trace hierarchy:

- **Trace** — one per Kiro CLI conversation
  - **Session** — grouped by `cwd`
  - **Events** — `agentSpawn`, `stop`
  - **Generations** — user prompts
  - **Spans** — tool use (`preToolUse` → `postToolUse`)
  - **Scores** — completion status (0–1)

<!-- TODO: replace with your actual trace screenshot, uploaded to langfuse.com images, and a public example trace link -->
![Example Kiro CLI trace in Langfuse](https://langfuse.com/images/cookbook/integration-kiro-cli/kiro-cli-example-trace.png)

[Example trace in Langfuse](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/TODO)

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more-js.mdx" -->